# Set up for dataset and model

Package installation, loading, and dataloaders. There's also a resnet18 model defined.

In [7]:
# !pip install tensorboardX

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from tqdm import tqdm

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

use_cuda = True
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.CIFAR10('cifar10_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.CIFAR10('cifar10_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



In [8]:

def tp_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return .5 * (x + delta) * (1 - ind1) * (1 - ind2) + x * ind2

def tp_smoothed_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return (x + delta) ** 2 / (4 * delta) * (1 - ind1) * (1 - ind2) + x * ind2

class Normalize(nn.Module):
    def __init__(self, mu, std):
        super(Normalize, self).__init__()
        self.mu, self.std = mu, std

    def forward(self, x):
        return (x - self.mu) / self.std

class IdentityLayer(nn.Module):
    def forward(self, inputs):
        return inputs
    
class PreActBlock(nn.Module):
    '''Pre-activation version of the BasicBlock.'''
    expansion = 1

    def __init__(self, in_planes, planes, bn, learnable_bn, stride=1, activation='relu'):
        super(PreActBlock, self).__init__()
        self.collect_preact = True
        self.activation = activation
        self.avg_preacts = []
        self.bn1 = nn.BatchNorm2d(in_planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not learnable_bn)
        self.bn2 = nn.BatchNorm2d(planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not learnable_bn)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=not learnable_bn)
            )

    def act_function(self, preact):
        if self.activation == 'relu':
            act = F.relu(preact)
        elif self.activation[:6] == '3prelu':
            act = tp_relu(preact, delta=float(self.activation.split('relu')[1]))
        elif self.activation[:8] == '3psmooth':
            act = tp_smoothed_relu(preact, delta=float(self.activation.split('smooth')[1]))
        else:
            assert self.activation[:8] == 'softplus'
            beta = int(self.activation.split('softplus')[1])
            act = F.softplus(preact, beta=beta)
        return act

    def forward(self, x):
        out = self.act_function(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x  # Important: using out instead of x
        out = self.conv1(out)
        out = self.conv2(self.act_function(self.bn2(out)))
        out += shortcut
        return out

class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, n_cls, cuda=True, half_prec=False,
        activation='relu', fts_before_bn=False, normal='none'):
        super(PreActResNet, self).__init__()
        self.bn = True
        self.learnable_bn = True  # doesn't matter if self.bn=False
        self.in_planes = 64
        self.avg_preact = None
        self.activation = activation
        self.fts_before_bn = fts_before_bn
        if normal == 'cifar10':
            self.mu = torch.tensor((0.4914, 0.4822, 0.4465)).view(1, 3, 1, 1)
            self.std = torch.tensor((0.2471, 0.2435, 0.2616)).view(1, 3, 1, 1)
        else:
            self.mu = torch.tensor((0.0, 0.0, 0.0)).view(1, 3, 1, 1)
            self.std = torch.tensor((1.0, 1.0, 1.0)).view(1, 3, 1, 1)
            print('no input normalization')
        if cuda:
            self.mu = self.mu.cuda()
            self.std = self.std.cuda()
        if half_prec:
            self.mu = self.mu.half()
            self.std = self.std.half()

        self.normalize = Normalize(self.mu, self.std)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not self.learnable_bn)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.bn = nn.BatchNorm2d(512 * block.expansion)
        self.linear = nn.Linear(512*block.expansion, n_cls)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, self.bn, self.learnable_bn, stride, self.activation))
            # layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, return_features=False):
        for layer in [*self.layer1, *self.layer2, *self.layer3, *self.layer4]:
            layer.avg_preacts = []

        out = self.normalize(x)
        out = self.conv1(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        if return_features and self.fts_before_bn:
            return out.view(out.size(0), -1)
        out = F.relu(self.bn(out))
        if return_features:
            return out.view(out.size(0), -1)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)

        return out


def PreActResNet18(n_cls, cuda=True, half_prec=False, activation='relu', fts_before_bn=False,
    normal='none'):
    #print('initializing PA RN-18 with act {}, normal {}'.format())
    return PreActResNet(PreActBlock, [2, 2, 2, 2], n_cls=n_cls, cuda=cuda, half_prec=half_prec,
        activation=activation, fts_before_bn=fts_before_bn, normal=normal)


# intialize the model
model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
model.eval()

no input normalization


PreActResNet(
  (normalize): Normalize()
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (layer1): Sequential(
    (0): PreActBlock(
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
    (1): PreActBlock(
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
  )
  (layer2): Sequential(
    (0): PreActBloc

# Implement the Attacks

Functions are given a simple useful signature that you can start with. Feel free to extend the signature as you see fit.

You may find it useful to create a 'batched' version of PGD that you can use to create the adversarial attack.

# Evaluate Single and Multi-Norm Robust Accuracy

In this section, we evaluate the model on the Linf and L2 attacks as well as union accuracy.

In [9]:
def pgd_linf_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True) 
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        # TODO: Calculate the loss
        loss = ce_loss(output, labels) 
        loss.backward()
        # TODO: compute the adv_x                                                                  
        adv_x = adv_x + eps_step * adv_x.grad.data.sign()
        # find delta, clamp with eps
        delta = adv_x - x
        delta = torch.clamp(delta, min=-eps, max=eps)
        adv_x = torch.clamp(x + delta, min=0, max=1).detach()
   
    return adv_x

In [10]:
def pgd_l2_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True) 
    for _ in range(k):
          adv_x.requires_grad_(True)
          model.zero_grad()
          output = model(adv_x)
          batch_size = x.size()[0]
          # TODO: Calculate the loss
          loss = ce_loss(output, labels)
          loss.backward()
          # TODO: compute the adv_x
          # find delta, clamp with eps, project delta to the l2 ball
          # HINT: https://github.com/Harry24k/adversarial-attacks-pytorch/blob/master/torchattacks/attacks/pgdl2.py 
          grad = adv_x.grad.data.sign()
          grad_norm = torch.norm(grad.view(batch_size, -1), p=2, dim=1)
          normalized_grad = grad / grad_norm.view(batch_size, 1, 1, 1)
          
          adv_x = adv_x.detach() + eps_step * normalized_grad
          
          delta = adv_x - x
          delta_norm = torch.norm(delta.view(batch_size, -1), p=2, dim=1)
          factor = torch.min(eps / delta_norm, torch.ones_like(delta_norm))
          delta = delta * factor.view(-1, 1, 1, 1)
          adv_x = torch.clamp(x + delta, min=0, max=1).detach()
   
    return adv_x

# Evaluate Single and Multi-Norm Robust Accuracy

In this section, we evaluate the model on the Linf and L2 attacks as well as union accuracy.

In [11]:
def test_model_on_single_attack(model, attack='pgd_linf', eps=0.1):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    tot_test_o, tot_acc_o = 0.0, 0.0
    k = 10
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        if attack == 'pgd_linf':
            # TODO: get x_adv untargeted pgd linf with eps, and eps_step=eps/4
            x_adv = pgd_linf_untargeted(model, x_batch, y_batch, k, eps, eps_step=eps/4)
        elif attack == 'pgd_l2':
            # TODO: get x_adv untargeted pgd l2 with eps, and eps_step=eps/4
            x_adv = pgd_l2_untargeted(model, x_batch, y_batch, k, eps, eps_step=eps/4)
        else:
            pass
        
        # get the testing accuracy and update tot_test and tot_acc
        with torch.no_grad():
            output = model(x_adv)
            pred = torch.max(output, dim=1)[1]
            tot_acc += (pred == y_batch).sum().item()
            tot_test += y_batch.size(0)
            
            output_o = model(x_batch)
            pred_o = torch.max(output_o, dim=1)[1]
            tot_acc_o += (pred_o == y_batch).sum().item()
            tot_test_o += y_batch.size(0)  
            
    print('Robust accuracy %.5lf' % (tot_acc/tot_test), f'on {attack} attack with eps = {eps}')
    print('Standard accuracy %.5lf' % (tot_acc_o/tot_test_o), f'on original attack with eps = {eps}')

## Single-Norm Robust Accuracy

In [ ]:
'''
# Evaluate on Linf attack with different models with eps = 8/255
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
# Evaluate on Linf attack with model 1 with eps = 8/255
test_model_on_single_attack(model, attack='pgd_linf', eps=8/255) 

model.load_state_dict(torch.load('models/pretr_L2.pth'))
# Evaluate on Linf attack with model 2 with eps = 8/255
test_model_on_single_attack(model, attack='pgd_linf', eps=8/255) 

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
# Evaluate on Linf attack with model 3 with eps = 8/255
test_model_on_single_attack(model, attack='pgd_linf', eps=8/255) 
'''

Evaluating: 100%|██████████| 157/157 [00:17<00:00,  8.90it/s]


Robust accuracy 0.51200 on pgd_linf attack with eps = 0.03137254901960784
Standard accuracy 0.82800 on original attack with eps = 0.03137254901960784


Evaluating: 100%|██████████| 157/157 [00:14<00:00, 10.87it/s]


Robust accuracy 0.30880 on pgd_linf attack with eps = 0.03137254901960784
Standard accuracy 0.88760 on original attack with eps = 0.03137254901960784


Evaluating: 100%|██████████| 157/157 [00:14<00:00, 10.81it/s]

Robust accuracy 0.49740 on pgd_linf attack with eps = 0.03137254901960784
Standard accuracy 0.81190 on original attack with eps = 0.03137254901960784


In [ ]:
'''
# Evaluate on L2 attack with different models with eps = 0.75
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
# Evaluate on Linf attack with model 1 with eps = 0.75
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75) 

model.load_state_dict(torch.load('models/pretr_L2.pth'))
# Evaluate on Linf attack with model 2 with eps = 0.75
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75) 

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
# Evaluate on Linf attack with model 3 with eps = 0.75
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75) 
'''

Evaluating:   0%|          | 0/157 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 157/157 [00:15<00:00, 10.47it/s]


Robust accuracy 0.70410 on pgd_l2 attack with eps = 0.75
Standard accuracy 0.82800 on original attack with eps = 0.75


Evaluating: 100%|██████████| 157/157 [00:14<00:00, 10.63it/s]


Robust accuracy 0.66940 on pgd_l2 attack with eps = 0.75
Standard accuracy 0.88760 on original attack with eps = 0.75


Evaluating: 100%|██████████| 157/157 [00:14<00:00, 10.55it/s]

Robust accuracy 0.69010 on pgd_l2 attack with eps = 0.75
Standard accuracy 0.81190 on original attack with eps = 0.75


## Multi-Norm Robust Accuracy

In [14]:
def test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    tot_test_o, tot_acc_o = 0.0, 0.0
    k = 10
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        # TODO: get x_adv_linf and x_adv_l2 untargeted pgd linf and l2 with eps, and eps_step=eps/4
        x_adv_linf = pgd_linf_untargeted(model, x_batch, y_batch, k, eps_linf, eps_step=eps_linf/4)
        x_adv_l2 = pgd_l2_untargeted(model, x_batch, y_batch, k, eps_l2, eps_step = eps_l2/4)
        
        ## calculate union accuracy: correct only if both attacks are correct
        
        out = model(x_adv_linf)
        pred_linf = torch.max(out, dim=1)[1]
        out = model(x_adv_l2)
        pred_l2 = torch.max(out, dim=1)[1]
        
        # TODO: get the testing accuracy with multi-norm robustness and update tot_test and tot_acc
        tot_acc += ((pred_linf == y_batch) & (pred_l2 == y_batch)).sum().item()
        tot_test += y_batch.size(0)
        
        output_o = model(x_batch)
        pred_o = torch.max(output_o, dim=1)[1]
        tot_acc_o += (pred_o == y_batch).sum().item()
        tot_test_o += y_batch.size(0)  
            
    print('Robust accuracy %.5lf' % (tot_acc/tot_test), f'on multi attacks')
    print('Standard accuracy %.5lf' % (tot_acc_o/tot_test_o), f'on original attack')

In [15]:
# Evaluate on multi-norm attacks with different models with eps_linf = 8./255, eps_l2 = 0.75
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
# Evaluate on multi attacks with model 1
test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)

model.load_state_dict(torch.load('models/pretr_L2.pth'))
# Evaluate on multi attacks with model 2
test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
# Evaluate on multi attacks with model 3
test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)

Evaluating: 100%|██████████| 157/157 [00:28<00:00,  5.52it/s]


Robust accuracy 0.51200 on multi attacks
Standard accuracy 0.82800 on original attack


Evaluating: 100%|██████████| 157/157 [00:27<00:00,  5.72it/s]


Robust accuracy 0.30880 on multi attacks
Standard accuracy 0.88760 on original attack


Evaluating: 100%|██████████| 157/157 [00:27<00:00,  5.70it/s]

Robust accuracy 0.49740 on multi attacks
Standard accuracy 0.81190 on original attack


Standard Accuracy Evaluation Function

In [16]:
def evaluate_standard_accuracy(model, test_loader, device):
    """Evaluate standard accuracy on clean test data"""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return 100. * correct / total

In [17]:
def evaluate_robust_accuracy(model, test_loader, device, eps, k=10, eps_step=None):
    """Evaluate robust accuracy against PGD attack"""
    if eps_step is None:
        eps_step = eps / 4
    
    model.eval()
    correct = 0
    total = 0
    
    for inputs, targets in tqdm(test_loader, desc='Evaluating robustness'):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Generate adversarial examples using your PGD function
        adv_inputs = pgd_linf_untargeted(model, inputs, targets, k, eps, eps_step)
        
        # Evaluate on adversarial examples
        with torch.no_grad():
            outputs = model(adv_inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return 100. * correct / total

In [18]:
def adversarial_training(model, train_loader, test_loader, device, 
                        epochs=10, lr=0.1, eps=8/255, k=7, eps_step=None):
    """
    PGD-based Adversarial Training
    
    Key: Train on adversarial examples (adv_x) with correct labels (y_batch)
    
    Args:
        model: Neural network model
        train_loader: Training data loader
        test_loader: Test data loader
        device: cuda or cpu
        epochs: Number of training epochs
        lr: Learning rate
        eps: Epsilon for PGD attack (perturbation budget)
        k: Number of PGD steps during training
        eps_step: Step size for each PGD iteration
    """
    if eps_step is None:
        eps_step = eps / 4
    
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, 
                                               milestones=[int(epochs*0.5), int(epochs*0.75)], 
                                               gamma=0.1)
    criterion = nn.CrossEntropyLoss()
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'standard_acc': [],
        'robust_acc': []
    }
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for batch_idx, (inputs, targets) in enumerate(pbar):
            inputs, targets = inputs.to(device), targets.to(device)
            model.eval()
            adv_inputs = pgd_linf_untargeted(model, inputs, targets, k, eps, eps_step)
            
            model.train()
            
            optimizer.zero_grad()
            outputs = model(adv_inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            # Track metrics
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            pbar.set_postfix({
                'Loss': f'{train_loss/(batch_idx+1):.3f}',
                'Acc': f'{100.*correct/total:.2f}%'
            })
        
        scheduler.step()
        
        # Record training metrics
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(100. * correct / total)
        
        # Evaluate after each epoch
        print(f'\nEpoch {epoch+1} Results:')
        standard_acc = evaluate_standard_accuracy(model, test_loader, device)
        robust_acc = evaluate_robust_accuracy(model, test_loader, device, eps, k=10, eps_step=eps/4)
        
        history['standard_acc'].append(standard_acc)
        history['robust_acc'].append(robust_acc)
        
        print(f'Standard Accuracy: {standard_acc:.2f}%')
        print(f'Robust Accuracy (eps={eps:.4f}): {robust_acc:.2f}%')
        print('-' * 60)
    
    return model, history

In [19]:
# Part a
epsilon_values = [0.01, 0.05, 0.1]
all_results = {}

print(f"Testing {len(epsilon_values)} epsilon values: {epsilon_values}")
print(f"Using k={5} PGD steps during training\n")

for eps in epsilon_values:
    print(f"\n{'='*80}")
    print(f"Training with epsilon = {eps:.4f}")
    print(f"{'='*80}\n")
    
    # Initialize new model for each epsilon
    model_adv = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
    
    # Train model with adversarial training
    trained_model, history = adversarial_training(
        model=model_adv,
        train_loader=train_loader,
        test_loader=test_loader,
        device=device,
        epochs=10,  # Increase to 50-100 for better results
        lr=0.1,
        eps=eps,
        k=10,  # Using k=10 PGD steps
        eps_step=eps/4
    )
    
    # Save the trained model with epsilon value in filename
    model_name = f'adv_trained_eps_{eps:.4f}.pth'
    torch.save(trained_model.state_dict(), model_name)
    print(f"\nModel saved as: {model_name}")
    
    # Store results with epsilon as key
    all_results[f'eps_{eps:.4f}'] = history
    
    # Final evaluation
    print(f"\n{'='*80}")
    print(f"Final Results for eps={eps:.4f}:")
    print(f"Standard Accuracy: {history['standard_acc'][-1]:.2f}%")
    print(f"Robust Accuracy: {history['robust_acc'][-1]:.2f}%")
    print(f"Accuracy Drop: {history['standard_acc'][-1] - history['robust_acc'][-1]:.2f}%")
    print(f"{'='*80}\n")

print("\n" + "="*80)
print("TRAINING COMPLETE FOR ALL EPSILON VALUES")
print("="*80)

Testing 3 epsilon values: [0.01, 0.05, 0.1]
Using k=5 PGD steps during training


Training with epsilon = 0.0100

no input normalization


Epoch 1/10: 100%|██████████| 782/782 [01:17<00:00, 10.12it/s, Loss=1.887, Acc=28.88%]



Epoch 1 Results:


Evaluating robustness: 100%|██████████| 157/157 [00:14<00:00, 11.19it/s]


Standard Accuracy: 40.03%
Robust Accuracy (eps=0.0100): 32.20%
------------------------------------------------------------


...

Standard Accuracy: 16.11%
Robust Accuracy (eps=0.1000): 14.31%
------------------------------------------------------------


Epoch 10/10: 100%|██████████| 782/782 [01:16<00:00, 10.16it/s, Loss=2.262, Acc=14.07%]



Epoch 10 Results:


Evaluating robustness: 100%|██████████| 157/157 [00:14<00:00, 11.06it/s]

Standard Accuracy: 16.22%
Robust Accuracy (eps=0.1000): 14.44%
------------------------------------------------------------

Model saved as: adv_trained_eps_0.1000.pth

Final Results for eps=0.1000:
Standard Accuracy: 16.22%
Robust Accuracy: 14.44%
Accuracy Drop: 1.78%


TRAINING COMPLETE FOR ALL EPSILON VALUES


In [22]:
# Train a standard (non-adversarial) model for comparison in Part (b)
print("Training standard model for comparison...")
standard_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
optimizer = optim.SGD(standard_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

# Standard training
num_epochs = 10
for epoch in range(num_epochs):
    standard_model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Standard Training {epoch+1}/{num_epochs}')
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = standard_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        pbar.set_postfix({
            'Loss': f'{train_loss/(len(pbar)):.3f}',
            'Acc': f'{100.*correct/total:.2f}%'
        })
    
    # Evaluate
    std_acc = evaluate_standard_accuracy(standard_model, test_loader, device)
    print(f'Epoch {epoch+1}: Standard Accuracy = {std_acc:.2f}%')

torch.save(standard_model.state_dict(), 'standard_trained.pth')
print("\nStandard model training complete!")
print(f"Final Standard Accuracy: {evaluate_standard_accuracy(standard_model, test_loader, device):.2f}%")

Training standard model for comparison...
no input normalization


Standard Training 1/10: 100%|██████████| 782/782 [00:16<00:00, 48.15it/s, Loss=1.665, Acc=38.28%]


Epoch 1: Standard Accuracy = 37.05%


Standard Training 2/10: 100%|██████████| 782/782 [00:15<00:00, 48.98it/s, Loss=1.285, Acc=53.14%]


Epoch 2: Standard Accuracy = 47.54%


...

Epoch 8: Standard Accuracy = 25.88%


Standard Training 9/10: 100%|██████████| 782/782 [00:15<00:00, 50.94it/s, Loss=0.792, Acc=72.55%]


Epoch 9: Standard Accuracy = 28.20%


Standard Training 10/10: 100%|██████████| 782/782 [00:15<00:00, 50.47it/s, Loss=0.770, Acc=73.29%]


Epoch 10: Standard Accuracy = 42.85%

Standard model training complete!
Final Standard Accuracy: 42.85%


In [25]:
#part b
# Cell: Part (b) - Compare Attacks on Original vs Adversarially Trained Models

# Load adversarially trained model - use epsilon = 0.05 (middle value from Part a)
test_eps = 0.05  # This matches one of your trained models

adv_trained_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
adv_trained_model.load_state_dict(torch.load(f'adv_trained_eps_{test_eps:.4f}.pth'))
adv_trained_model.eval()

print(f"Loaded adversarially trained model (trained with ε={test_eps:.4f})")
print(f"Comparing against standard model\n")

def compare_attacks_on_models(standard_model, adv_trained_model, test_loader, device):
    """
    Part (b): Compare effectiveness of FGSM attack on original vs adversarially trained models
    """
    
    # Test with FGSM only (1-step PGD) - matching Part (a) epsilon values
    attack_configs = [
        {'name': 'FGSM (1-step)', 'k': 1, 'eps_values': [0.01, 0.05, 0.1]},
    ]
    
    results = {
        'standard': {'clean_acc': 0, 'attacks': {}},
        'adv_trained': {'clean_acc': 0, 'attacks': {}}
    }
    
    # Evaluate clean accuracy
    print("="*80)
    print("PART (b): FGSM Attack Comparison - Original vs Adversarially Trained Model")
    print("="*80)
    print("\nEvaluating Standard Accuracy...")
    results['standard']['clean_acc'] = evaluate_standard_accuracy(standard_model, test_loader, device)
    results['adv_trained']['clean_acc'] = evaluate_standard_accuracy(adv_trained_model, test_loader, device)
    
    print(f"Standard Model - Clean Accuracy: {results['standard']['clean_acc']:.2f}%")
    print(f"Adv Trained Model (ε={test_eps:.2f}) - Clean Accuracy: {results['adv_trained']['clean_acc']:.2f}%\n")
    
    # Test FGSM attack
    for attack_config in attack_configs:
        attack_name = attack_config['name']
        k = attack_config['k']
        
        print(f"\n{'='*80}")
        print(f"Testing {attack_name}")
        print(f"{'='*80}")
        
        results['standard']['attacks'][attack_name] = []
        results['adv_trained']['attacks'][attack_name] = []
        
        for eps in attack_config['eps_values']:
            print(f"\nEpsilon = {eps:.4f}")
            
            # Attack standard model
            std_robust_acc = evaluate_robust_accuracy(
                standard_model, test_loader, device, 
                eps=eps, k=k, eps_step=eps  # For FGSM, eps_step = eps
            )
            results['standard']['attacks'][attack_name].append(std_robust_acc)
            
            # Attack adversarially trained model
            adv_robust_acc = evaluate_robust_accuracy(
                adv_trained_model, test_loader, device,
                eps=eps, k=k, eps_step=eps  # For FGSM, eps_step = eps
            )
            results['adv_trained']['attacks'][attack_name].append(adv_robust_acc)
            
            # Calculate effectiveness (accuracy drop)
            std_drop = results['standard']['clean_acc'] - std_robust_acc
            adv_drop = results['adv_trained']['clean_acc'] - adv_robust_acc
            
            print(f"  Standard Model: {std_robust_acc:.2f}% (drop: {std_drop:.2f}%)")
            print(f"  Adv Trained Model: {adv_robust_acc:.2f}% (drop: {adv_drop:.2f}%)")
            print(f"  Improvement: +{adv_robust_acc - std_robust_acc:.2f}%")
            if adv_drop > 0:
                print(f"  Attack is {std_drop/adv_drop:.2f}x MORE effective on standard model")
            else:
                print(f"  Adversarially trained model is perfectly robust at this epsilon!")
    
    return results

# Run comparison
comparison_results = compare_attacks_on_models(standard_model, adv_trained_model, test_loader, device)

no input normalization
Loaded adversarially trained model (trained with ε=0.0500)
Comparing against standard model

PART (b): FGSM Attack Comparison - Original vs Adversarially Trained Model

Evaluating Standard Accuracy...
Standard Model - Clean Accuracy: 42.85%
Adv Trained Model (ε=0.05) - Clean Accuracy: 40.06%


Testing FGSM (1-step)

Epsilon = 0.0100


Evaluating robustness: 100%|██████████| 157/157 [00:03<00:00, 48.50it/s]


  Standard Model: 6.57% (drop: 36.28%)
  Adv Trained Model: 36.85% (drop: 3.21%)
  Improvement: +30.28%
  Attack is 11.30x MORE effective on standard model

Epsilon = 0.0500


Evaluating robustness: 100%|██████████| 157/157 [00:03<00:00, 49.63it/s]


  Standard Model: 7.64% (drop: 35.21%)
  Adv Trained Model: 25.01% (drop: 15.05%)
  Improvement: +17.37%
  Attack is 2.34x MORE effective on standard model

Epsilon = 0.1000


Evaluating robustness: 100%|██████████| 157/157 [00:03<00:00, 51.97it/s]

  Standard Model: 9.94% (drop: 32.91%)
  Adv Trained Model: 14.08% (drop: 25.98%)
  Improvement: +4.14%
  Attack is 1.27x MORE effective on standard model
